# Q16: What does an embedding refer to in natural language processing?
**Topic:** NLP | **Difficulty:** Hard | **Time:** 18 min
**Sheet:** Grind75ML

---

## Question
What does an embedding refer to in natural language processing?

## Detailed Answer

An **embedding** is a dense, low-dimensional, learned vector representation that captures semantic meaning of a discrete input (word, sentence, document).

### Why Embeddings?
- **One-hot encoding** is sparse and high-dimensional (vocab size ~50K+), with no semantic information
- Embeddings compress this to dense vectors (100-768d) where **similar items are close in vector space**

### Types of Embeddings:

#### 1. Word Embeddings (Static)
Same vector for a word regardless of context.

**Word2Vec** (Mikolov et al., 2013):
- **Skip-gram**: Predict context words from center word
- **CBOW**: Predict center word from context words
- Trained with negative sampling or hierarchical softmax
- Famous property: king - man + woman ≈ queen

**GloVe** (Pennington et al., 2014):
- Factorizes the word co-occurrence matrix
- Objective: $J = \sum_{i,j} f(X_{ij})(w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$

**FastText** (Bojanowski et al., 2017):
- Extends Word2Vec with subword (character n-gram) information
- Handles out-of-vocabulary words

#### 2. Contextual Embeddings (Dynamic)
Different vector for same word in different contexts.

- "bank" in "river bank" vs "bank account" → different vectors
- **ELMo**: Bidirectional LSTM
- **BERT**: Bidirectional Transformer encoder
- **GPT**: Unidirectional Transformer decoder

#### 3. Sentence Embeddings
- **Mean pooling** of word/token embeddings
- **[CLS] token** from BERT
- **Sentence-BERT**: Siamese network fine-tuned for similarity
- **Instructor embeddings**: Task-specific with instruction prefix

### Embedding Properties
| Property | Description |
|----------|-------------|
| Semantic similarity | cosine(king, queen) > cosine(king, car) |
| Analogies | king - man + woman ≈ queen |
| Clustering | Similar concepts cluster together |
| Transfer learning | Pre-trained embeddings transfer to downstream tasks |
| Dimensionality | Typically 100-768 dimensions |

In [ ]:
import numpy as np


class SimpleWord2Vec:
    """Simplified Skip-gram Word2Vec implementation."""
    
    def __init__(self, vocab_size: int, embedding_dim: int, lr: float = 0.01):
        self.W_embed = np.random.randn(vocab_size, embedding_dim) * 0.1  # input embeddings
        self.W_context = np.random.randn(vocab_size, embedding_dim) * 0.1  # context embeddings
        self.lr = lr
    
    @staticmethod
    def sigmoid(x: np.ndarray) -> np.ndarray:
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def train_pair(self, center_idx: int, context_idx: int, label: int) -> float:
        """Train on one (center, context, label) pair. label=1 for positive, 0 for negative."""
        # Forward
        center_vec = self.W_embed[center_idx]
        context_vec = self.W_context[context_idx]
        score = self.sigmoid(np.dot(center_vec, context_vec))
        
        # Loss (binary cross-entropy)
        eps = 1e-12
        loss = -(label * np.log(score + eps) + (1 - label) * np.log(1 - score + eps))
        
        # Gradients
        grad = score - label
        self.W_embed[center_idx] -= self.lr * grad * context_vec
        self.W_context[context_idx] -= self.lr * grad * center_vec
        
        return loss
    
    def get_embedding(self, word_idx: int) -> np.ndarray:
        return self.W_embed[word_idx]
    
    def similarity(self, idx1: int, idx2: int) -> float:
        v1, v2 = self.W_embed[idx1], self.W_embed[idx2]
        return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


# --- Demo on toy corpus ---
corpus = 'the cat sat on the mat the dog chased the cat dogs and cats are pets'
words = corpus.split()
vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}

model = SimpleWord2Vec(len(vocab), embedding_dim=10, lr=0.1)

# Training: skip-gram with window=2, negative sampling
window = 2
for epoch in range(50):
    total_loss = 0
    for i, word in enumerate(words):
        center = word_to_idx[word]
        for j in range(max(0, i-window), min(len(words), i+window+1)):
            if i == j: continue
            context = word_to_idx[words[j]]
            total_loss += model.train_pair(center, context, 1)  # positive
            neg = np.random.randint(len(vocab))  # negative sample
            total_loss += model.train_pair(center, neg, 0)

print(f'Vocab: {vocab}')
print(f'\nSimilarity(cat, dog): {model.similarity(word_to_idx["cat"], word_to_idx["dog"]):.4f}')
print(f'Similarity(cat, mat): {model.similarity(word_to_idx["cat"], word_to_idx["mat"]):.4f}')
print(f'\nEmbedding for "cat": {model.get_embedding(word_to_idx["cat"]).round(3)}')

## Key Takeaways
- Embeddings capture **semantic meaning** in dense vectors — fundamental to modern NLP
- **Static** (Word2Vec, GloVe) vs **contextual** (BERT, GPT) — contextual is SOTA
- **Pre-trained embeddings** are standard — training from scratch is only for specialized domains
- The embedding layer is the first layer of virtually every NLP model